# 실전 13-1: 기본 추론(CoT) 및 다수결 검증(Self-Consistency)

## 1. Chain-of-Thought (CoT) 란?
- 복잡한 문제를 LLM에게 주었을 때 곧바로 답을 내놓게 하면 틀릴 확률이 높습니다.
- "단계별로 생각해서(Think step by step) 풀어라"라고 지시(Zero-shot CoT)하거나,
- 사람이 직접 푸는 과정(추론 과정)을 예시로 보여주는 것(Few-shot CoT)만으로도 모델의 추론(Reasoning) 능력이 비약적으로 상승합니다.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# 온도를 0으로 설정하여 일관된 결과를 유도합니다.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 2. Zero-shot CoT vs Few-shot CoT 비교
아주 살짝 꼬여있는 논리 수학 문제를 내봅시다.

In [2]:
question = "주차장에 자동차와 오토바이가 섞여 있습니다. 바퀴의 총합은 100개이고, 차량의 총합은 30대입니다. 자동차와 오토바이는 각각 몇 대인가요?"

print("=== [1. 일반 프롬프트 (그냥 묻기)] ===")
normal_prompt = ChatPromptTemplate.from_template("다음 문제를 풀고 답만 말해줘: {question}")
normal_chain = normal_prompt | llm
print(normal_chain.invoke({"question": question}).content)

print("\n=== [2. Zero-shot CoT (단계별로 생각하기)] ===")
zero_cot_prompt = ChatPromptTemplate.from_template("다음 문제를 차근차근 단계별로 논리적으로 생각(Think step by step)해서 푼 다음, 마지막에 최종 답을 제시해줘: {question}")
zero_cot_chain = zero_cot_prompt | llm
print(zero_cot_chain.invoke({"question": question}).content)

print("\n=== [3. Few-shot CoT (풀이 과정 예시 제공)] ===")
# 비슷한 유형의 문제 풀이 과정을 미리 보여주어, LLM이 '어떻게 추론해야 하는지' 모방하게 만듭니다.
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 수학 논리 전문가입니다. 주어진 예시처럼 방정식을 세워 차근차근 푸세요."),
    ("user", "닭과 토끼가 섞여 있습니다. 머리는 10개, 다리는 26개입니다. 각각 몇 마리인가요?"),
    ("assistant", "닭을 x, 토끼를 y라고 합시다.\n1. x + y = 10 (머리 수)\n2. 2x + 4y = 26 (다리 수)\n3. 첫 번째 식에서 x = 10 - y 입니다.\n4. 두 번째 식에 대입: 2(10 - y) + 4y = 26 => 20 - 2y + 4y = 26 => 2y = 6 => y = 3\n5. 따라서 토끼는 3마리, 닭은 7마리입니다."),
    ("user", "{question}")
])
few_cot_chain = few_shot_prompt | llm
print(few_cot_chain.invoke({"question": question}).content)

=== [1. 일반 프롬프트 (그냥 묻기)] ===
자동차는 20대, 오토바이는 10대입니다.

=== [2. Zero-shot CoT (단계별로 생각하기)] ===
이 문제를 해결하기 위해 자동차와 오토바이의 수를 변수로 설정하고, 주어진 조건을 바탕으로 방정식을 세워보겠습니다.

1. **변수 설정**:
   - 자동차의 수를 \( x \)라고 하고,
   - 오토바이의 수를 \( y \)라고 하겠습니다.

2. **주어진 조건**:
   - 차량의 총합: \( x + y = 30 \) (1)
   - 바퀴의 총합: 자동차는 4개의 바퀴, 오토바이는 2개의 바퀴를 가지고 있으므로, \( 4x + 2y = 100 \) (2)

3. **방정식 정리**:
   - (1)번 방정식에서 \( y \)를 \( 30 - x \)로 표현할 수 있습니다.
   - 이를 (2)번 방정식에 대입해 보겠습니다.

   \[
   4x + 2(30 - x) = 100
   \]

4. **방정식 풀기**:
   - 위의 식을 전개하면:
   \[
   4x + 60 - 2x = 100
   \]
   - 정리하면:
   \[
   2x + 60 = 100
   \]
   - 양변에서 60을 빼면:
   \[
   2x = 40
   \]
   - 양변을 2로 나누면:
   \[
   x = 20
   \]

5. **오토바이 수 구하기**:
   - \( x = 20 \)을 (1)번 방정식에 대입하여 \( y \)를 구합니다:
   \[
   20 + y = 30
   \]
   - 따라서:
   \[
   y = 10
   \]

6. **결과 확인**:
   - 자동차 수: \( x = 20 \)
   - 오토바이 수: \( y = 10 \)
   - 바퀴 수 확인: \( 4 \times 20 + 2 \times 10 = 80 + 20 = 100 \) (조건 만족)

최종적으로, 주차장에는 자동차가 20대, 오토바이가 10대 있습니다. 

**답: 자동차 20대, 오토바이 

## 3. Self-Consistency (자기 일관성 다수결)
- 가장 똑똑한 모델도 가끔 계산 실수를 하거나 환각을 일으킵니다.
- 이를 방지하기 위해 **모델 온도를 약간 높여(temperature=0.7) 동일한 문제를 5번 다르게 풀게 시킨 뒤, 가장 많이 나온 답안을 최종 정답으로 채택**하는 기법입니다.

In [4]:
from collections import Counter
import re

# 창의적(다양한 경로의 추론)으로 5번 풀기 위해 온도를 올립니다.
creative_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
sc_chain = zero_cot_prompt | creative_llm

answers = []
print("=== [Self-Consistency: 5번 반복 풀이 시작] ===")
for i in range(5):
    response = sc_chain.invoke({"question": question}).content
    # 정답을 파싱 (보통 '자동차 20대, 오토바이 10대' 처럼 나옴)
    # 단순화를 위해 전체 텍스트에서 '20'과 '10'이라는 숫자를 추출하여 튜플로 묶습니다.
    numbers = re.findall(r'\d+', response)
    # 마지막으로 도출된 자동차, 오토바이 수(일반적으로 글의 끝부분에 위치)
    if len(numbers) >= 2:
        final_answer = f"자동차 {numbers[-2]}대, 오토바이 {numbers[-1]}대"
    else:
        final_answer = "해석 불가"
        
    answers.append(final_answer)
    print(f"[{i+1}번째 시도 결론]: {final_answer}")

# 다수결 투표
counter = Counter(answers)
majority_answer = counter.most_common(1)[0][0]
print(f"\n🏆 [최종 다수결 정답]: {majority_answer} (득표: {counter.most_common(1)[0][1]} / 5)")

=== [Self-Consistency: 5번 반복 풀이 시작] ===
[1번째 시도 결론]: 자동차 20대, 오토바이 10대
[2번째 시도 결론]: 자동차 20대, 오토바이 10대
[3번째 시도 결론]: 자동차 20대, 오토바이 10대
[4번째 시도 결론]: 자동차 20대, 오토바이 10대
[5번째 시도 결론]: 자동차 20대, 오토바이 10대

🏆 [최종 다수결 정답]: 자동차 20대, 오토바이 10대 (득표: 5 / 5)


## 4. 결론
- 엉성하게 물어보면 엉성하게 대답하지만, **과정(Step)**을 밟게 하면 똑똑해집니다 (CoT).
- 한 명의 천재에게 묻는 것보다, **한 명의 천재에게 5번 물어보고 다수결**을 내는 것이 훨씬 더 강력한 안전망이 됩니다 (Self-Consistency).